## Setup SageMaker FeatureStore

In [2]:
import boto3
import sagemaker

original_boto3_version = boto3.__version__
%pip install 'boto3>1.17.21'

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Note: you may need to restart the kernel to use updated packages.


In [4]:
from sagemaker.session import Session

region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

bucket = feature_store_session.default_bucket()
bucket

'sagemaker-us-east-1-730335647345'

In [6]:
s3 = boto3.client("s3", region_name=region)

s3_private_path_csv = f's3://{bucket}/'
s3_key_housing = 'homework3-1/housing.csv'
s3_key_gmaps = 'homework3-1/housing_gmaps_data_raw.csv'

with open('housing.csv') as f:
    s3.put_object(Bucket=bucket, Key=s3_key_housing, Body=f.read())
    
with open('housing_gmaps_data_raw.csv') as f:
    s3.put_object(Bucket=bucket, Key=s3_key_gmaps, Body=f.read())

## Inspect Dataset

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io


housing_data_object = s3.get_object(
    Bucket=bucket, Key=s3_key_housing
)
gmaps_data_object = s3.get_object(
    Bucket=bucket, Key=s3_key_gmaps
)

housing_data = pd.read_csv(io.BytesIO(housing_data_object["Body"].read()))
gmaps_data = pd.read_csv(io.BytesIO(gmaps_data_object["Body"].read()))

In [32]:
housing_data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [21]:
gmaps_data.head()

,street_number,route,locality-political,administrative_area_level_2-political,administrative_area_level_1-political,country-political,postal_code,address,longitude,latitude,...,establishment-natural_feature,airport-establishment-point_of_interest,political-sublocality-sublocality_level_1,administrative_area_level_3-political,post_box,establishment-light_rail_station-point_of_interest-transit_station,establishment-point_of_interest,aquarium-establishment-park-point_of_interest-tourist_attraction-zoo,campground-establishment-lodging-park-point_of_interest-rv_park-tourist_attraction,cemetery-establishment-park-point_of_interest
0,3130,Grizzly Peak Boulevard,Berkeley,Alameda County,California,United States,94705.0,"3130 Grizzly Peak Blvd, Berkeley, CA 94705, USA",-122.23,37.88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005,Tunnel Road,Oakland,Alameda County,California,United States,94611.0,"2005 Tunnel Rd, Oakland, CA 94611, USA",-122.22,37.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6886,Chabot Road,Oakland,Alameda County,California,United States,94618.0,"6886 Chabot Rd, Oakland, CA 94618, USA",-122.24,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6365,Florio Street,Oakland,Alameda County,California,United States,94618.0,"6365 Florio St, Oakland, CA 94618, USA",-122.25,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5407,Bryant Avenue,Oakland,Alameda County,California,United States,94618.0,"5407 Bryant Ave, Oakland, CA 94618, USA",-122.25,37.84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Feature Engineering

In [78]:
# Feature Engineering

housing_data = housing_data.round(5)
gmaps_data = gmaps_data.round(5)

housing_data = housing_data.fillna(0)
gmaps_data = gmaps_data.fillna(0)

# Join datasets
neighbourhood_data = pd.merge(housing_data, gmaps_data, how="inner", on=["latitude", "longitude"])

# Primary key
neighbourhood_data.rename(columns={"neighborhood-political": "neighborhood"}, inplace=True)

# Prepare mean total bedrooms per postal code for bedrooms per household
postal_mean = neighbourhood_data.groupby("postal_code")["total_bedrooms"].mean()
neighbourhood_data["total_bedrooms"] = neighbourhood_data["total_bedrooms"].fillna(neighbourhood_data["postal_code"].map(postal_mean))

# Aggregate median house value, housing median age, households and ocean proximity
neighbourhood_data = (
    neighbourhood_data.groupby("neighborhood", as_index=False)
      .agg(
          median_house_value=("median_house_value", "mean"),
          housing_median_age=("housing_median_age", "mean"),
          total_households=("households", "mean"),
          ocean_proximity=("ocean_proximity", lambda x: x.mode().iat[0]),
          households_sum=("households", "sum"),
          total_bedrooms_sum=("total_bedrooms", "sum")
      )
)

# Bedrooms per household
neighbourhood_data["bedrooms_per_household"] = (
    neighbourhood_data["total_bedrooms_sum"] / neighbourhood_data["households_sum"]
)
neighbourhood_data.drop(columns=["households_sum", "total_bedrooms_sum"], inplace=True)

# Ensure total households is int
neighbourhood_data["total_households"] = np.ceil(neighbourhood_data["total_households"]).astype("Int64")

# One hot encode the ocean proximity
encoded_ocean_proximity = pd.get_dummies(neighbourhood_data["ocean_proximity"], prefix="ocean_proximity")
neighbourhood_data = pd.concat(
    [neighbourhood_data, encoded_ocean_proximity], axis=1
)
neighbourhood_data = neighbourhood_data.rename(
    columns={"ocean_proximity_<1H OCEAN": "ocean_proximity_lt1H_OCEAN",
             "ocean_proximity_NEAR BAY": "ocean_proximity_NEAR_BAY",
             "ocean_proximity_NEAR OCEAN": "ocean_proximity_NEAR_OCEAN"
            }
)
neighbourhood_data.drop(columns=['ocean_proximity'], inplace=True)

neighbourhood_data

,neighborhood,median_house_value,housing_median_age,total_households,bedrooms_per_household,ocean_proximity_lt1H_OCEAN,ocean_proximity_INLAND,ocean_proximity_NEAR_BAY,ocean_proximity_NEAR_OCEAN
0,0,192184.221306,25.634880,495,1.079076,False,True,False,False
1,28 Palms,222200.000000,25.000000,923,1.017335,True,False,False,False
2,Acorn Industrial,81300.000000,52.000000,147,1.659864,False,False,True,False
3,Adams Hill,250733.333333,39.500000,494,1.053680,True,False,False,False
4,Agua Mansa Industrial Corridor,112300.000000,17.000000,516,1.102713,False,True,False,False
...,...,...,...,...,...,...,...,...,...
1302,Woodside Plaza,346150.000000,32.750000,820,1.018598,False,False,False,True
1303,Wrigley Heights,225300.000000,32.666667,492,1.061653,False,False,False,True
1304,Wyndham,101200.000000,23.000000,420,0.971429,False,True,False,False
1305,Ygnacio Valley,351600.000000,23.333333,548,1.017032,False,False,True,False


# Ingest Data into FeatureStore

In [81]:
from time import gmtime, strftime, sleep

neighbourhood_feature_group_name = "neighbourhood_data-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

In [82]:
from sagemaker.feature_store.feature_group import FeatureGroup

neighbourhood_feature_group = FeatureGroup(
    name=neighbourhood_feature_group_name, sagemaker_session=feature_store_session
)

In [85]:
import time

current_time_sec = int(round(time.time()))


def cast_columns(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")
        if data_frame.dtypes[label] == "bool":
            data_frame[label] = data_frame[label].astype("int")


# cast object dtype to string. The SageMaker FeatureStore Python SDK will then map the string dtype to String feature type.
cast_columns(neighbourhood_data)

# record identifier and event time feature names
record_identifier_feature_name = "neighborhood"
event_time_feature_name = "EventTime"

# append EventTime feature
neighbourhood_data[event_time_feature_name] = pd.Series(
    [current_time_sec] * len(neighbourhood_data), dtype="float64"
)

# load feature definitions to the feature group. SageMaker FeatureStore Python SDK will auto-detect the data schema based on input data.
neighbourhood_feature_group.load_feature_definitions(data_frame=neighbourhood_data)
# output is suppressed


[FeatureDefinition(feature_name='neighborhood', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='median_house_value', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='housing_median_age', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='total_households', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='bedrooms_per_household', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='ocean_proximity_lt1H_OCEAN', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='ocean_proximity_INLAND', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='ocean_proximity_NEAR_BAY', feature_type=<

In [89]:
from sagemaker import get_execution_role

def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


prefix = 'neighbourhood_assignment'
neighbourhood_feature_group.create(
    s3_uri=f"s3://{bucket}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=get_execution_role(),
    enable_online_store=True,
)

wait_for_feature_group_creation_complete(feature_group=neighbourhood_feature_group)

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup neighbourhood_data-feature-group-23-01-55-03 successfully created.


In [91]:
neighbourhood_feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:730335647345:feature-group/neighbourhood_data-feature-group-23-01-55-03',
 'FeatureGroupName': 'neighbourhood_data-feature-group-23-01-55-03',
 'RecordIdentifierFeatureName': 'neighborhood',
 'EventTimeFeatureName': 'EventTime',
 'FeatureDefinitions': [{'FeatureName': 'neighborhood',
   'FeatureType': 'String'},
  {'FeatureName': 'median_house_value', 'FeatureType': 'Fractional'},
  {'FeatureName': 'housing_median_age', 'FeatureType': 'Fractional'},
  {'FeatureName': 'total_households', 'FeatureType': 'Integral'},
  {'FeatureName': 'bedrooms_per_household', 'FeatureType': 'Fractional'},
  {'FeatureName': 'ocean_proximity_lt1H_OCEAN', 'FeatureType': 'Integral'},
  {'FeatureName': 'ocean_proximity_INLAND', 'FeatureType': 'Integral'},
  {'FeatureName': 'ocean_proximity_NEAR_BAY', 'FeatureType': 'Integral'},
  {'FeatureName': 'ocean_proximity_NEAR_OCEAN', 'FeatureType': 'Integral'},
  {'FeatureName': 'EventTime', 'FeatureType': 'Fractional'}

In [92]:
neighbourhood_feature_group.ingest(data_frame=neighbourhood_data, max_workers=3, wait=True)

IngestionManagerPandas(feature_group_name='neighbourhood_data-feature-group-23-01-55-03', feature_definitions={'neighborhood': {'FeatureName': 'neighborhood', 'FeatureType': 'String'}, 'median_house_value': {'FeatureName': 'median_house_value', 'FeatureType': 'Fractional'}, 'housing_median_age': {'FeatureName': 'housing_median_age', 'FeatureType': 'Fractional'}, 'total_households': {'FeatureName': 'total_households', 'FeatureType': 'Integral'}, 'bedrooms_per_household': {'FeatureName': 'bedrooms_per_household', 'FeatureType': 'Fractional'}, 'ocean_proximity_lt1H_OCEAN': {'FeatureName': 'ocean_proximity_lt1H_OCEAN', 'FeatureType': 'Integral'}, 'ocean_proximity_INLAND': {'FeatureName': 'ocean_proximity_INLAND', 'FeatureType': 'Integral'}, 'ocean_proximity_NEAR_BAY': {'FeatureName': 'ocean_proximity_NEAR_BAY', 'FeatureType': 'Integral'}, 'ocean_proximity_NEAR_OCEAN': {'FeatureName': 'ocean_proximity_NEAR_OCEAN', 'FeatureType': 'Integral'}, 'EventTime': {'FeatureName': 'EventTime', 'Featur

In [93]:
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

record_identifier_value = 'Brooktree'

featurestore_runtime.get_record(
    FeatureGroupName=neighbourhood_feature_group_name,
    RecordIdentifierValueAsString=record_identifier_value,
)

{'ResponseMetadata': {'RequestId': '969fbb55-70f6-4780-82b2-130946a5e914',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '969fbb55-70f6-4780-82b2-130946a5e914',
   'content-type': 'application/json',
   'content-length': '895',
   'date': 'Tue, 23 Sep 2025 02:04:38 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'neighborhood', 'ValueAsString': 'Brooktree'},
  {'FeatureName': 'median_house_value', 'ValueAsString': '257400.0'},
  {'FeatureName': 'housing_median_age', 'ValueAsString': '9.0'},
  {'FeatureName': 'total_households', 'ValueAsString': '1438'},
  {'FeatureName': 'bedrooms_per_household', 'ValueAsString': '0.0'},
  {'FeatureName': 'ocean_proximity_lt1H_OCEAN', 'ValueAsString': '1'},
  {'FeatureName': 'ocean_proximity_INLAND', 'ValueAsString': '0'},
  {'FeatureName': 'ocean_proximity_NEAR_BAY', 'ValueAsString': '0'},
  {'FeatureName': 'ocean_proximity_NEAR_OCEAN', 'ValueAsString': '0'},
  {'FeatureName': 'EventTime', 'ValueAsString': '1758592712.0'}]}

In [107]:
neighbourhood_query = neighbourhood_feature_group.athena_query()

neighbourhood_table = neighbourhood_query.table_name


query_string = f"""
SELECT * FROM {neighbourhood_table} WHERE neighborhood IN ('Brooktree', 'Fisherman''s Wharf',  'Los Osos')
"""
print("Running " + query_string)

neighbourhood_query.run(
    query_string=query_string,
    output_location="s3://" + bucket + "/" + prefix + "/query_results/",
)
neighbourhood_query.wait()
dataset = neighbourhood_query.as_dataframe()

dataset

Running 
SELECT * FROM neighbourhood_data_feature_group_23_01_55_03_1758592847 WHERE neighborhood IN ('Brooktree', 'Fisherman''s Wharf',  'Los Osos')



,neighborhood,median_house_value,housing_median_age,total_households,bedrooms_per_household,ocean_proximity_lt1h_ocean,ocean_proximity_inland,ocean_proximity_near_bay,ocean_proximity_near_ocean,eventtime,write_time,api_invocation_time,is_deleted
0,Los Osos,221612.5,15.375,612,1.050266,0,0,0,1,1.758593e+09,2025-09-23 02:07:04.157,2025-09-23 02:02:00.000,False
1,Brooktree,257400.0,9.000,1438,0.000000,1,0,0,0,1.758593e+09,2025-09-23 02:07:04.155,2025-09-23 02:02:00.000,False
2,Fisherman's Wharf,500001.0,52.000,250,1.268000,0,0,1,0,1.758593e+09,2025-09-23 02:07:04.141,2025-09-23 02:02:06.000,False
